In [1]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/bse_block_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today
today = datetime.today()

api_date = today.strftime("%d/%m/%Y")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"block_{file_date}.csv"
)

try:

    url = (
        "https://api.bseindia.com/BseIndiaAPI/api/"
        "BulknBlockBETADwnld/w"
        f"?DealType=2"
        "&sc_code="
        f"&FDate={api_date}"
        f"&TDate={api_date}"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    # no block deals today
    if response.text.strip() == "":

        df = pd.DataFrame()

    else:

        df = pd.read_csv(
            io.StringIO(response.text)
        )

        # standardize column names
        df.columns = [
            "Deal_Date",
            "Security_Code",
            "Company",
            "Client_Name",
            "Deal_Type",
            "Quantity",
            "Price"
        ]

    # save file
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)

    if rows == 0:
        message = "No data yet"
    else:
        message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# logging
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_block",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, d6983510-3713-429f-bbab-5c10e107b1ef, 3, Finished, Available, Finished, False)

SUCCESS
Rows: 0
No data yet
